# Repository inventory and selection

**Purpose:** Explain the MicrosoftDocs repository inventory, scoring, and selected documentation sources used to build tokenizer training/evaluation corpora.

**Inputs (committed result/provenance files read):**
- [`inventory/repo_inventory.csv`](../inventory/repo_inventory.csv)
- [`inventory/repo_scores.csv`](../inventory/repo_scores.csv)
- [`inventory/SELECTION.md`](../inventory/SELECTION.md)

**Outputs:** Rendered inventory/scoring summaries in this notebook only; no files are written.

**Expected runtime:** ~3 seconds (loads committed CSV/JSON/parquet files only).

**Traceability links:** [`FINAL_REPORT.md`](../results/reports/FINAL_REPORT.md) and [`CLAIM_LEDGER.md`](../publication/CLAIM_LEDGER.md).

**Execution notes:** Analysis-only notebook. It resolves `ROOT` by walking upward to `plan.md`, uses no absolute paths or machine identifiers, does not download corpora, does not retrain tokenizers, and has no hidden state. Randomness: none used. Exploratory analysis: none; cells below directly present committed result files.


In [1]:
import json
from pathlib import Path

import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
while not (ROOT / 'plan.md').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

def rel(path):
    path = Path(path)
    try:
        return str(path.relative_to(ROOT))
    except ValueError:
        return str(path)

def note_missing(path):
    print(f'NOTE: optional input is missing or empty: {path}. The notebook continues without it.')

def load_csv(path):
    p = ROOT / path
    if not p.exists():
        note_missing(path)
        return pd.DataFrame()
    return pd.read_csv(p)

def load_parquet(path):
    p = ROOT / path
    if not p.exists():
        note_missing(path)
        return pd.DataFrame()
    try:
        return pd.read_parquet(p)
    except Exception as exc:
        print(f'NOTE: could not read optional parquet {path}: {exc}')
        return pd.DataFrame()

def load_json(path):
    p = ROOT / path
    if not p.exists():
        note_missing(path)
        return {}
    return json.loads(p.read_text())

def load_text(path):
    p = ROOT / path
    if not p.exists():
        note_missing(path)
        return ''
    return p.read_text()

def safe_read_json_path(path):
    try:
        return json.loads(Path(path).read_text())
    except Exception as exc:
        print(f'NOTE: could not read {rel(path)}: {exc}')
        return {}

print(f'Repository root: {rel(ROOT)}')


Repository root: .


## Repository inventory

Authoritative files: [`repo_inventory.csv`](../inventory/repo_inventory.csv), [`repo_scores.csv`](../inventory/repo_scores.csv), and [`SELECTION.md`](../inventory/SELECTION.md).

In [2]:
inventory = load_csv('inventory/repo_inventory.csv')
scores = load_csv('inventory/repo_scores.csv')
selection = load_text('inventory/SELECTION.md')

In [3]:
if not inventory.empty:
    print(f'Repositories inventoried: {len(inventory):,}')
    cols = ['full_name', 'language', 'license_spdx', 'archived', 'size_kb', 'pushed_at']
    display(inventory[cols].head(15))
else:
    note_missing('inventory/repo_inventory.csv')

Repositories inventoried: 809


,full_name,language,license_spdx,archived,size_kb,pushed_at
0,MicrosoftDocs/Virtualization-Documentation,PowerShell,CC-BY-4.0,False,100898,2026-06-11T23:03:20Z
1,MicrosoftDocs/visualfsharpdocs,F#,CC-BY-4.0,True,50377,2024-11-26T06:44:04Z
2,MicrosoftDocs/PowerShell-Docs,PowerShell,NOASSERTION,False,107126,2026-07-08T22:06:04Z
3,MicrosoftDocs/windows-driver-docs,OpenSCAD,CC-BY-4.0,False,428771,2026-07-08T21:46:35Z
4,MicrosoftDocs/ATADocs,NaN,CC-BY-4.0,True,132831,2025-02-12T20:28:05Z
5,MicrosoftDocs/Azure-RMSDocs,CSS,NOASSERTION,False,133788,2026-05-08T14:50:07Z
6,MicrosoftDocs/EMDocs,NaN,CC-BY-4.0,True,31208,2024-11-26T06:46:09Z
7,MicrosoftDocs/MIMDocs,Shell,CC-BY-4.0,False,38286,2026-01-27T18:32:03Z
8,MicrosoftDocs/WSL,NaN,NOASSERTION,False,53786,2026-07-07T21:34:57Z
9,MicrosoftDocs/navdevitpro-content-pr,NaN,CC-BY-4.0,False,39350,2024-11-05T08:18:33Z


In [4]:
if not scores.empty:
    print(f'Scored candidate sources: {len(scores):,}')
    cols = ['full_name', 'weighted_total', 'domain_relevance', 'cross_domain_value', 'low_duplication_risk', 'notes']
    display(scores.sort_values('weighted_total', ascending=False)[cols].head(15))
else:
    note_missing('inventory/repo_scores.csv')

Scored candidate sources: 13


,full_name,weighted_total,domain_relevance,cross_domain_value,low_duplication_risk,notes
0,MicrosoftDocs/azure-docs,0.9075,0.20,0.08,0.0275,ANCHOR only. Current public active Azure docs;...
3,MicrosoftDocs/sql-docs,0.9070,0.18,0.08,0.0400,Selected. docs/relational-databases verified w...
1,MicrosoftDocs/architecture-center,0.8730,0.18,0.09,0.0425,Selected. 502 Markdown files under docs (10.0 ...
2,MicrosoftDocs/azure-devops-docs,0.8250,0.18,0.08,0.0450,Selected. docs/pipelines verified with 393 Mar...
7,MicrosoftDocs/visualstudio-docs,0.8250,0.12,0.08,0.0400,Strong alternate broad developer corpus: docs ...
8,MicrosoftDocs/cpp-docs,0.8100,0.10,0.08,0.0450,Strong alternate programming-language/referenc...
6,MicrosoftDocs/PowerShell-Docs,0.8050,0.16,0.08,0.0450,Deferred despite high utility. reference/7.4 v...
12,MicrosoftDocs/memdocs,0.8030,0.14,0.08,0.0400,Endpoint management docs appear relevant and a...
10,MicrosoftDocs/dynamics-365-unified-operations-...,0.7950,0.14,0.07,0.0400,Large business-app docs corpus: articles has 6...
9,MicrosoftDocs/SupportArticles-docs,0.7800,0.12,0.08,0.0400,Large support/troubleshooting corpus: support ...


In [5]:
if selection:
    print(selection[:3000])
else:
    note_missing('inventory/SELECTION.md')

# Repository selection for tokenizer-efficiency evaluation

Generated: 2026-07-10. Inventory source: `gh api --paginate --slurp 'orgs/MicrosoftDocs/repos?per_page=100&type=public'`. Path verification used `gh api repos/<repo>/contents/<path>` plus subtree `git/trees/<sha>?recursive=1`; all listed repo paths below had `truncated=false` and contain `.md` files.

## Azure anchor (training corpus)

Full `MicrosoftDocs/azure-docs` is huge (`size_kb=27046986`, about 25.8 GiB) and should not be fully checked out under the 25 GiB source-disk budget. Pin and acquire only the existing sparse training paths.

| repo | branch | pinned commit | sparse paths | verified Markdown | est. cleaned Markdown | hard-filter result |
|---|---|---|---|---:|---:|---|
| MicrosoftDocs/azure-docs | main | `80e8be2fca4160d1ff2528ae84bbf96f4a90d46e` | `articles/storage`, `articles/virtual-network`, `articles/application-gateway`, `articles/frontdoor`, `articles/expressroute`, `articles/app-service`, `articles/azure-